# XLM-R + Syllable/Char BiLSTM-CRF Exact ABSA

Notebook này dùng kiến trúc gần với notebook tham khảo: XLM-R contextual embedding + token/syllable embedding + char BiLSTM + word-level BiLSTM + CRF. Khác biệt quan trọng là checkpoint được chọn bằng dev exact character-span F1 của repo mình, và output được ghi thành JSONL để `src.absa.evaluate` chấm được.

## Install Dependencies

In [ ]:
!pip install -q transformers sentencepiece protobuf scikit-learn

## Check GPU Compatibility

Nếu Kaggle cấp Tesla P100 và PyTorch báo không hỗ trợ `sm_60`, hãy đổi Accelerator sang T4/T4x2 rồi chạy lại. Train XLM-R trên CPU sẽ rất chậm nên notebook dừng sớm trong trường hợp GPU không tương thích.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Please enable a Kaggle GPU accelerator.")

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU:", gpu_name)
print("CUDA capability:", capability)
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

if capability[0] < 7:
    raise RuntimeError(
        "This Kaggle runtime's PyTorch build does not support this GPU well enough for training. "
        "Switch the Kaggle accelerator from P100 to T4/T4x2, then restart and rerun the notebook."
    )

## Configure Project and Output Paths

In [ ]:
import os
from datetime import datetime

REPO_URL = "https://github.com/ngocvo2511/NLP.git"
PROJECT_DIR = "/kaggle/working/NLP"
RUN_NAME = datetime.utcnow().strftime("xlmr_bilstm_crf_exact_%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"/kaggle/working/NLP_outputs/{RUN_NAME}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!python -m src.absa.validate_data --data-dir data
print("Output root:", OUTPUT_ROOT)
print("Run name:", RUN_NAME)

## Train and Evaluate on Dev

Cell này chỉ dùng `train.jsonl` để train và dùng `dev.jsonl` để chọn best checkpoint. Chưa dùng test, để tránh tune theo test.

In [ ]:
!python -m src.absa.train_xlmr_bilstm_crf_exact \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-bilstm-crf-exact" \
  --model-name FacebookAI/xlm-roberta-base \
  --epochs 30 \
  --batch-size 16 \
  --max-length 256 \
  --transformer-lr 2e-5 \
  --head-lr 1e-3 \
  --warmup-ratio 0.1 \
  --syllable-dim 100 \
  --char-emb-dim 50 \
  --char-hidden-dim 50 \
  --lstm-hidden-size 400 \
  --dropout 0.33 \
  --seed 42

!cat "$OUTPUT_ROOT/xlmr-bilstm-crf-exact/dev_metrics.json"

## Optional Final Test Evaluation

Chỉ chạy cell này sau khi đã chốt cấu hình bằng dev. Script sẽ train lại, chọn best bằng dev, rồi chấm test đúng một lần trên best checkpoint.

In [ ]:
# FINAL_OUTPUT_ROOT = f"{OUTPUT_ROOT}_final_test"
# os.makedirs(FINAL_OUTPUT_ROOT, exist_ok=True)
# !python -m src.absa.train_xlmr_bilstm_crf_exact \
#   --data-dir data \
#   --output-dir "$FINAL_OUTPUT_ROOT/xlmr-bilstm-crf-exact" \
#   --model-name FacebookAI/xlm-roberta-base \
#   --epochs 30 \
#   --batch-size 16 \
#   --max-length 256 \
#   --transformer-lr 2e-5 \
#   --head-lr 1e-3 \
#   --warmup-ratio 0.1 \
#   --syllable-dim 100 \
#   --char-emb-dim 50 \
#   --char-hidden-dim 50 \
#   --lstm-hidden-size 400 \
#   --dropout 0.33 \
#   --seed 42 \
#   --eval-test
# !cat "$FINAL_OUTPUT_ROOT/xlmr-bilstm-crf-exact/test_metrics.json"

## Export Compact Results

In [ ]:
import json
import shutil
from pathlib import Path

compact_dir = Path('/kaggle/working/absa_xlmr_bilstm_crf_exact_outputs')
compact_dir.mkdir(parents=True, exist_ok=True)

for path in Path(OUTPUT_ROOT).rglob('*'):
    if path.name in {'dev_metrics.json', 'dev_predictions.jsonl', 'history.json', 'metadata.json', 'test_metrics.json', 'test_predictions.jsonl'}:
        target = compact_dir / path.relative_to(OUTPUT_ROOT)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

archive = shutil.make_archive(str(compact_dir), 'zip', compact_dir)
print('Compact output directory:', compact_dir)
print('Compact zip:', archive)

metrics_path = Path(OUTPUT_ROOT) / 'xlmr-bilstm-crf-exact' / 'dev_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Dev exact span micro:', metrics['micro'])